# Показ 3. Тематическое моделирование экзит-интервью

Запускаем модели на синтетическом датасете и смотрим, какие темы (причины увольнения) они находят.

1. **LDA** — классическая вероятностная модель, которую предлагает кейс.
2. **NMF** — разложение матрицы TF-IDF.
3. **Qwen3-Embedding-4B + KMeans** — смысловые векторы нейросети и кластеризация (подход BERTopic).

Для каждой модели считаем совпадение с эталонными причинами (ARI, F1) — датасет синтетический, поэтому правильный ответ известен.

In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.max_colwidth', 100)

from src.evaluation.model_comparison import load_corpus
from src.evaluation.metrics import external_metrics, majority_mapping, per_label_report
from src.models import LDATopicModel, NMFTopicModel, EmbeddingTopicModel, Encoder

texts, tokens, y = load_corpus()  # очистка, лемматизация, без пустых ответов и дублей
print(f'Ответов: {len(texts)}')
pd.DataFrame({'текст для e5': texts[:3], 'леммы для LDA/NMF': [' '.join(t) for t in tokens[:3]]})

Ответов: 459


,текст для e5,леммы для LDA/NMF
0,"Нет менторства и наставничества, новые навыки получить негде.",менторство наставничество новый навык получить негде
1,Выгорела полностью. Не могу больше.,выгореть полностью
2,"Не обеспечили нормальными условиями для удалённой работы — ни техники, ни компенсации.",обеспечить нормальный условие удаленка техника компенсация


In [2]:
def show(result, name):
    """Темы модели: размер, ключевые слова и какой эталонной причине тема соответствует."""
    m = external_metrics(y, result.labels)
    print(f"{name}: тем {result.n_topics}, ARI {m['ARI']:.2f}, macro-F1 {m['macro_F1']:.2f}")
    info = result.topic_info(8)
    info['эталон'] = info['topic'].map(majority_mapping(y, result.labels))
    return info[['topic', 'size', 'эталон', 'keywords']]

## 1. LDA

Каждый ответ — смесь тем, каждая тема — распределение слов. Обучается на частотах лемм.

In [3]:
lda = LDATopicModel(n_topics=10).fit(texts, tokens)
show(lda, 'LDA')

LDA: тем 10, ARI 0.02, macro-F1 0.17


,topic,size,эталон,keywords
0,4,65,management,"руководитель, время, задача, отдел, процесс, сделать, обещать, постоянный"
1,9,49,career,"руководитель, много, новый, команда, расти, направление, понять, задача"
2,8,48,personal,"новый, час, почти, команда, хватать, личный, жизнь, начальник"
3,6,47,conditions,"плохой, склад, коллектив, условие, команда, постоянный, время, техника"
4,1,45,schedule,"график, руководство, постоянный, год, подходить, невозможно, менять, обстоятельство"
5,5,44,schedule,"офис, сторона, удаленка, рост, режим, здоровье, город, отменить"
6,0,43,content,"коллега, система, проект, дать, план, день, ребенок, идти"
7,3,43,personal,"день, пять, согласование, задача, должность, оборудование, неделя, повышение"
8,2,42,salary,"зарплата, нормальный, смена, компенсация, найти, проблема, нельзя, год"
9,7,33,salary,"деньга, переработка, позиция, платить, совсем, ощущение, команда, рабочий"


Слова внутри тем выглядят связно, но темы почти не совпадают с настоящими причинами: в ответе в среднем 5 лемм, и LDA не хватает данных, чтобы оценить смесь тем.

## 2. NMF

In [4]:
nmf = NMFTopicModel(n_topics=10).fit(texts, tokens)
show(nmf, 'NMF')

NMF: тем 10, ARI 0.10, macro-F1 0.24


,topic,size,эталон,keywords
0,9,74,schedule,"график, здоровье, подходить, личный, склад, смена, час, постоянный"
1,6,58,team,"команда, новый, почти, общение, старый, комфортно, больший, найти"
2,4,56,salary,"зарплата, платить, устраивать, нормальный, должность, вырасти, расти, собеседование"
3,5,52,content,"задача, расти, творческий, ответственность, сделать, соответствовать, рост, ощущение"
4,1,46,schedule,"день, рабочий, неделя, однообразный, офис, пять, удаленка, коллега"
5,0,43,management,"руководитель, сотрудник, ошибка, коллектив, повышение, деньга, процесс, поощрять"
6,7,37,career,"отдел, обещать, начальник, постоянный, принимать, любой, информация, ждать"
7,8,37,management,"руководство, сторона, коллега, постоянный, тяжело, процесс, семейный, обстоятельство"
8,3,35,personal,"время, год, обещать, техника, сделать, чувствовать, поступить, забирать"
9,2,21,team,"плохой, условие, коллектив, склад, холодно, начальник, труд, сквозняк"


## 3. Эмбеддинги Qwen3-Embedding-4B + KMeans

Нейросеть `Qwen3-Embedding-4B` превращает ответ в вектор смысла. Инструкция «Identify the main reason why the employee is leaving the company» говорит модели обращать внимание на причину ухода, а не на стиль ответа. KMeans делит векторы на 10 групп. Ключевые слова — c-TF-IDF по леммам.

In [5]:
qwen = EmbeddingTopicModel(n_topics=10, encoder=Encoder('Qwen/Qwen3-Embedding-4B'), clustering='kmeans')
best = qwen.fit(texts, tokens)
show(best, 'Qwen3-Embedding-4B + KMeans')

Qwen3-Embedding-4B + KMeans: тем 10, ARI 0.68, macro-F1 0.83


,topic,size,эталон,keywords
0,5,60,salary,"зарплата, платить, деньга, компенсация, бонус, оклад, оплата, получать"
1,4,59,personal,"здоровье, переезжать, уезжать, дом, идти, семья, ребенок, сменить"
2,7,53,career,"рост, повышение, развитие, обещать, позиция, должность, обучение, перспектива"
3,2,49,team,"коллектив, команда, коллега, атмосфера, отношение, сложиться, общение, сотрудник"
4,3,46,processes,"процесс, согласование, система, меняться, нельзя, много, половина, отчет"
5,9,44,schedule,"офис, график, удаленка, час, дорога, день, смена, подходить"
6,6,42,management,"руководитель, руководство, начальник, любой, сторона, заменить, ноль, контроль"
7,8,40,workload,"задача, переработка, ночь, постоянный, оставаться, норма, объем, выходной"
8,1,37,content,"задача, делать, проект, рутина, интересный, скучно, однообразный, творческий"
9,0,29,conditions,"условие, плохой, болеть, выдавать, оборудование, техника, склад, нормальный"


Типичные ответы каждой темы — то, что HR увидит в интерфейсе:

In [6]:
for t in best.topic_info()['topic']:
    print(f'Тема {t}:', ' | '.join(best.representative_docs(texts, t, 2)))

Тема 5: Оплата труда низкая. Других претензий к компании нет, коллектив хороший. | Уровень оплаты ниже среднего по городу, это все знают, поэтому текучка и большая.
Тема 4: По семейным обстоятельствам. | Выхожу замуж и переезжаю к мужу в Новосибирск.
Тема 7: Хотел вырасти до тимлида, но в компании нет такой роли в нашем направлении! | Получил второе высшее, но компания никак это не учла, остался на той же позиции.
Тема 2: Токсичный коллектив, постоянные сплетни за спиной. | коллектив токсичный, сплетни, интриги
Тема 3: Половина времени уходит на то, чтобы выяснить, кто должен сделать задачу. | Нет единой системы постановки задач, всё в чатах, почте и устно, постоянно что-то теряется!
Тема 9: Отменили удалёнку, а я уже переехала за город. ездить по два часа в одну сторону не готова. | график не устраивает и далеко ездить
Тема 6: Руководитель не защищает команду перед высшим руководством, всегда сдаёт своих. | Нет уважения со стороны начальства. Обращаются как к прислуге.
Тема 8: Сплошны

Качество по каждой причине:

In [7]:
per_label_report(y, best.labels).round(2)

,precision,recall,f1,support
label,,,,
career,0.77,0.91,0.84,45
conditions,0.93,0.63,0.75,43
content,0.86,0.70,0.77,46
management,0.88,0.80,0.84,46
personal,0.76,0.98,0.86,46
processes,0.80,0.82,0.81,45
salary,0.82,0.98,0.89,50
schedule,0.89,0.87,0.88,45
team,0.88,0.96,0.91,45


## 4. Сравнение всех моделей

Полная таблица: `python -m src.evaluation.model_comparison` (результаты в `docs/results/`). Устойчивость — средний ARI между пятью запусками на разных 80% данных.

In [8]:
res = pd.read_csv('../docs/results/models_k10.csv')
res.sort_values('ARI', ascending=False)[['модель', 'тем', 'ARI', 'macro_F1', 'c_v', 'diversity', 'stability']].round(3)

,модель,тем,ARI,macro_F1,c_v,diversity,stability
4,Qwen3-Embedding-4B + kmeans,10,0.675,0.830,0.383,0.970,0.798
6,Qwen3-Embedding-4B + hdbscan + UMAP5,14,0.580,0.779,0.388,0.971,0.704
5,Qwen3-Embedding-4B + agglomerative,10,0.574,0.710,0.342,0.940,0.723
12,multilingual-e5-large-instruct + hdbscan + UMAP5,11,0.510,0.629,0.360,0.946,0.409
10,multilingual-e5-large-instruct + kmeans,10,0.424,0.624,0.382,0.950,0.511
7,Qwen3-Embedding-0.6B + kmeans,10,0.411,0.616,0.387,0.920,0.498
9,Qwen3-Embedding-0.6B + hdbscan + UMAP5,9,0.346,0.536,0.354,0.944,0.537
11,multilingual-e5-large-instruct + agglomerative,10,0.313,0.487,0.370,0.950,0.506
24,FRIDA + hdbscan + UMAP5,14,0.292,0.566,0.388,0.879,0.476
15,multilingual-e5-large + hdbscan + UMAP5,14,0.270,0.505,0.406,0.886,0.452


In [9]:
from src.evaluation.plots import plot_coherence_vs_ari
plot_coherence_vs_ari(res);

## Выводы

- Классические модели «мешка слов» на коротких ответах почти не находят реальные причины: «мало платят» и «низкий оклад» не имеют общих слов.
- Эмбеддинги решают эту проблему, но важны две вещи: инструкция (e5-large 0.22 → e5-large-instruct 0.42) и размер модели (Qwen3-Embedding-0.6B 0.41 → 4B 0.68).
- Лучшая модель, Qwen3-Embedding-4B + KMeans, выполнила все цели: ARI 0.68, macro-F1 0.83, устойчивость 0.80. Каждая её тема соответствует одной настоящей причине.
- Связность тем (c_v) не отражает реального качества: у LDA она самая высокая. Поэтому модель выбираем по совпадению с эталоном на синтетике, а связность используем только как дополнительный сигнал.